In [1]:
# 正常框架
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20,256),
                    nn.ReLU(),
                    nn.Linear(256,10))
X=torch.rand(2,20)
print(X)
net(X)

tensor([[0.2596, 0.4477, 0.3759, 0.6182, 0.9987, 0.6260, 0.0651, 0.9217, 0.1025,
         0.3573, 0.7704, 0.3462, 0.8153, 0.7961, 0.6812, 0.0149, 0.5153, 0.5874,
         0.9369, 0.4688],
        [0.0466, 0.8826, 0.1694, 0.2834, 0.2213, 0.6802, 0.0380, 0.8102, 0.7208,
         0.1871, 0.2604, 0.3808, 0.4439, 0.3594, 0.7469, 0.0190, 0.1074, 0.9709,
         0.7239, 0.8159]])


tensor([[-0.1710,  0.1192,  0.0271, -0.2205, -0.2150,  0.0262,  0.1271, -0.0502,
         -0.3043,  0.1223],
        [-0.2447,  0.2284,  0.0722, -0.0923, -0.1878,  0.0594,  0.0382, -0.0237,
         -0.3263, -0.0104]], grad_fn=<AddmmBackward0>)

In [2]:
#任何一个层或者神经网络，都是Module的子类
#自定义快
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden=nn.Linear(20,256)
        self.out=nn.Linear(256,10)
    def forward(self,X):
        return self.out(F.relu(self.hidden(X)))

# 使用
net=MLP()
net(X)

tensor([[ 0.0696,  0.0134,  0.0344, -0.1295, -0.0019,  0.2034, -0.1378, -0.2727,
          0.0506, -0.0556],
        [ 0.0900, -0.0910, -0.0437, -0.1355, -0.0737,  0.0821, -0.0795, -0.2373,
         -0.0795, -0.0406]], grad_fn=<AddmmBackward0>)

In [3]:
#自定义实现顺序块
class mySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for block in args:
            self._modules[block]=block
    def forward(self,X):
        for block in self._modules.values():
            X=block(X)
        return X

net=mySequential(nn.Linear(20,256),
                    nn.ReLU(),
                    nn.Linear(256,10))
net(X)

tensor([[ 0.0418, -0.2065,  0.4580,  0.2275,  0.2581,  0.0408, -0.2495,  0.2439,
         -0.1055,  0.1303],
        [ 0.0070, -0.2016,  0.3381,  0.1797,  0.1707,  0.0623, -0.2814,  0.2097,
         -0.1035,  0.0557]], grad_fn=<AddmmBackward0>)

In [4]:
#另外一种写法
class MySequential(nn.Module):
    def __init__(slef,*args): #__init__函数将每个模块逐个添加到有序字典_modules中
        super().__init__()
        for idx,moodule in enumerate(args):
            # module是Module子类的一个实例
            # 变量 _modules中，_modules的类型是OrderedDict
            self._modules[str(idx)]=module
    def forward(self,X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

net=mySequential(nn.Linear(20,256),
                    nn.ReLU(),
                    nn.Linear(256,10))
net(X)        

tensor([[-0.0984, -0.0538, -0.0488, -0.1122,  0.0126, -0.1697, -0.1118,  0.0100,
         -0.1893, -0.1410],
        [-0.0724, -0.0682, -0.0906,  0.0050,  0.0299, -0.0773,  0.0082, -0.0936,
         -0.2130,  0.0349]], grad_fn=<AddmmBackward0>)

In [10]:
# 自定义块
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变(常量)，不会被反向传播
        self.rand_weight = torch.rand((20,20),requires_grad=False)
        self.linear = nn.Linear(20,20)
    def forward(self,X):
        X=self.linear(X)
        X=F.relu(torch.mm(X,self.rand_weight)+1)
        X=self.linear(X)
        while X.abs().sum()>1:
            X/=2
        return X.sum()
net = FixedHiddenMLP()
net(X)


tensor(-0.0178, grad_fn=<SumBackward0>)

In [11]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)
    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP()) #混合搭配各种组合块的方法
chimera(X)


tensor(0.1167, grad_fn=<SumBackward0>)